# 01 — Run rule-sensitivity simulations

This notebook runs the formal **rule-sensitivity experiment** for the KlimSta game model and saves every experimental condition separately.

The experiment varies two factors:

### Card access

| Access rule | Cards offered |
| --- | --- |
| `category_1x3` | 3 physical cards from 1 randomly selected category |
| `category_2x3` | 3 physical cards from each of 2 randomly selected categories |
| `random_10` | 10 random physical cards |
| `random_20` | 20 random physical cards |
| `random_30` | 30 random physical cards |

### Passing behaviour

At every decision with at least one playable card, the simulated player may end the round with probability:

`0.0`, `0.1`, `0.25`, or `0.5`.

This gives **5 × 4 = 20 experimental conditions**.

The notebook is intended to live in:

```text
Versionen/paper_draft_v1/notebooks/
```

Run it with the repository root as the working directory.


## 1. Setup

The model itself stays in `model/`. This notebook only defines and runs the experiment.

`REUSE_EXISTING = True` makes the batch resumable: a condition is loaded from disk when matching result files already exist. Set `FORCE_RERUN = True` to regenerate everything.


In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import time

import pandas as pd

from model.simulate import run_simulation


VERSION = Path("Versionen/paper_draft_v1")
EXPERIMENT_DIR = VERSION / "results" / "rule_sensitivity"

N_GAMES = 10_000
SEED = 42
LOG_CHOICES = True

REUSE_EXISTING = True
FORCE_RERUN = True

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Experimental design

The design is kept directly in the notebook so the exact conditions used for the paper remain visible and easy to change.

The category-based rules and the unrestricted random draws test different forms of limited choice and should not be interpreted as a single continuous "number of cards" scale.


In [ ]:
PASS_PROBABILITIES = [0.0, 0.1, 0.25]

ACCESS_RULES = [
    {
        "name": "category_1x3",
        "rule": "random_categories",
        "draw_n": 7,
        "category_count": 1,
        "cards_per_category": 3,
    },
    {
        "name": "category_2x3",
        "rule": "random_categories",
        "draw_n": 7,
        "category_count": 2,
        "cards_per_category": 3,
    },
    {
        "name": "random_10",
        "rule": "random_draw",
        "draw_n": 10,
        "category_count": 1,
        "cards_per_category": 3,
    },
    {
        "name": "random_20",
        "rule": "random_draw",
        "draw_n": 20,
        "category_count": 1,
        "cards_per_category": 3,
    },
    {
        "name": "random_30",
        "rule": "random_draw",
        "draw_n": 30,
        "category_count": 1,
        "cards_per_category": 3,
    },
]

design = pd.DataFrame([
    {
        **access,
        "pass_probability": pass_probability,
        "experiment": f"{access['name']}_pass_{pass_probability:g}",
    }
    for access in ACCESS_RULES
    for pass_probability in PASS_PROBABILITIES
])

design.to_csv(
    EXPERIMENT_DIR / "design.csv",
    index=False,
)

design


,name,rule,draw_n,category_count,cards_per_category,pass_probability,experiment
0,category_1x3,random_categories,7,1,3,0.00,category_1x3_pass_0
1,category_1x3,random_categories,7,1,3,0.10,category_1x3_pass_0.1
2,category_1x3,random_categories,7,1,3,0.25,category_1x3_pass_0.25
3,category_1x3,random_categories,7,1,3,0.50,category_1x3_pass_0.5
4,category_2x3,random_categories,7,2,3,0.00,category_2x3_pass_0
5,category_2x3,random_categories,7,2,3,0.10,category_2x3_pass_0.1
6,category_2x3,random_categories,7,2,3,0.25,category_2x3_pass_0.25
7,category_2x3,random_categories,7,2,3,0.50,category_2x3_pass_0.5
8,random_10,random_draw,10,1,3,0.00,random_10_pass_0
9,random_10,random_draw,10,1,3,0.10,random_10_pass_0.1


## 3. Run the batch

Each condition is saved immediately to:

```text
results/rule_sensitivity/<experiment>/
    games.parquet
    plays.parquet
    config.json
```

This means an interrupted batch does not lose completed simulations.

For an existing condition, the notebook checks that the saved number of games matches `N_GAMES`. Matching runs are reused; incomplete or incompatible runs are regenerated.


In [4]:
results = []
run_log = []

batch_start = time.perf_counter()

for _, condition in design.iterrows():

    experiment = condition["experiment"]
    run_dir = EXPERIMENT_DIR / experiment
    games_path = run_dir / "games.parquet"
    plays_path = run_dir / "plays.parquet"
    config_path = run_dir / "config.json"

    run_dir.mkdir(exist_ok=True)

    can_reuse = (
        REUSE_EXISTING
        and not FORCE_RERUN
        and games_path.exists()
        and (plays_path.exists() or not LOG_CHOICES)
    )

    if can_reuse:
        games = pd.read_parquet(games_path)
        plays = (
            pd.read_parquet(plays_path)
            if plays_path.exists()
            else pd.DataFrame()
        )

        # Do not silently reuse a run with the wrong sample size.
        can_reuse = len(games) == N_GAMES

    if can_reuse:
        print(f"{experiment}: loaded {len(games):,} existing games")
        status = "loaded"
        runtime = 0.0

    else:
        print(f"\n--- {experiment} ---")
        start = time.perf_counter()

        games, plays = run_simulation(
            VERSION,
            n_games=N_GAMES,
            rule=condition["rule"],
            draw_n=int(condition["draw_n"]),
            category_count=int(condition["category_count"]),
            cards_per_category=int(condition["cards_per_category"]),
            pass_probability=float(condition["pass_probability"]),
            seed=SEED,
            save=False,
            log_choices=LOG_CHOICES,
        )

        runtime = time.perf_counter() - start
        status = "simulated"

    # Store condition metadata also when an existing run is loaded.
    games["experiment"] = experiment
    games["access_rule"] = condition["name"]
    games["pass_probability"] = float(condition["pass_probability"])

    if not plays.empty:
        plays["experiment"] = experiment
        plays["access_rule"] = condition["name"]
        plays["pass_probability"] = float(condition["pass_probability"])

    # Save/re-save the condition in the current result format.
    games.to_parquet(games_path, index=False)

    if LOG_CHOICES:
        plays.to_parquet(plays_path, index=False)

    config = {
        "experiment": experiment,
        "access_rule": condition["name"],
        "rule": condition["rule"],
        "draw_n": int(condition["draw_n"]),
        "category_count": int(condition["category_count"]),
        "cards_per_category": int(condition["cards_per_category"]),
        "pass_probability": float(condition["pass_probability"]),
        "n_games": N_GAMES,
        "seed": SEED,
        "log_choices": LOG_CHOICES,
    }

    config_path.write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    results.append((games, plays))

    run_log.append({
        "experiment": experiment,
        "status": status,
        "games": len(games),
        "decisions": len(plays),
        "runtime_s": runtime,
    })

    # Persist the log after every condition.
    pd.DataFrame(run_log).to_csv(
        EXPERIMENT_DIR / "run_log.csv",
        index=False,
    )

print(
    f"\nBatch complete in "
    f"{time.perf_counter() - batch_start:,.1f} s"
)



--- category_1x3_pass_0 ---
10,000/10,000 games | 3,665 games/s

--- category_1x3_pass_0.1 ---
10,000/10,000 games | 3,466 games/s

--- category_1x3_pass_0.25 ---
10,000/10,000 games | 4,167 games/s

--- category_1x3_pass_0.5 ---
10,000/10,000 games | 4,678 games/s

--- category_2x3_pass_0 ---
10,000/10,000 games | 2,119 games/s

--- category_2x3_pass_0.1 ---
10,000/10,000 games | 2,183 games/s

--- category_2x3_pass_0.25 ---
10,000/10,000 games | 2,356 games/s

--- category_2x3_pass_0.5 ---
10,000/10,000 games | 4,021 games/s

--- random_10_pass_0 ---
10,000/10,000 games | 3,215 games/s

--- random_10_pass_0.1 ---
10,000/10,000 games | 2,919 games/s

--- random_10_pass_0.25 ---
10,000/10,000 games | 3,373 games/s

--- random_10_pass_0.5 ---
10,000/10,000 games | 7,189 games/s

--- random_20_pass_0 ---
10,000/10,000 games | 1,894 games/s

--- random_20_pass_0.1 ---
10,000/10,000 games | 2,119 games/s

--- random_20_pass_0.25 ---
10,000/10,000 games | 3,101 games/s

--- random_20_pass_

## 4. Combine the saved conditions

The individual run folders remain the authoritative per-condition results. For convenient analysis, the notebook also writes combined Parquet files.

These are what the second notebook loads.


In [5]:
all_games = pd.concat(
    [games for games, _ in results],
    ignore_index=True,
)

all_plays = pd.concat(
    [plays for _, plays in results if not plays.empty],
    ignore_index=True,
)

all_games.to_parquet(
    EXPERIMENT_DIR / "all_games.parquet",
    index=False,
)

all_plays.to_parquet(
    EXPERIMENT_DIR / "all_plays.parquet",
    index=False,
)

print(f"{len(all_games):,} games saved")
print(f"{len(all_plays):,} logged decisions saved")


200,000 games saved
1,900,385 logged decisions saved


## 5. Batch check

This final table is only a run-control check. Interpretation belongs in the second notebook.

With `N_GAMES = 10_000`, every one of the 20 conditions should contain exactly 10,000 games.


In [6]:
batch_check = (
    all_games
    .groupby(["experiment", "access_rule", "pass_probability"])
    .size()
    .rename("games")
    .reset_index()
)

batch_check


,experiment,access_rule,pass_probability,games
0,category_1x3_pass_0,category_1x3,0.00,10000
1,category_1x3_pass_0.1,category_1x3,0.10,10000
2,category_1x3_pass_0.25,category_1x3,0.25,10000
3,category_1x3_pass_0.5,category_1x3,0.50,10000
4,category_2x3_pass_0,category_2x3,0.00,10000
5,category_2x3_pass_0.1,category_2x3,0.10,10000
6,category_2x3_pass_0.25,category_2x3,0.25,10000
7,category_2x3_pass_0.5,category_2x3,0.50,10000
8,random_10_pass_0,random_10,0.00,10000
9,random_10_pass_0.1,random_10,0.10,10000


In [7]:
assert len(batch_check) == len(design)
assert (batch_check["games"] == N_GAMES).all()

print("All experimental conditions are complete.")


All experimental conditions are complete.
